# 02 | Optimization and decision trees

**Question:** Can we derive gradients and split criteria, then verify them independently?

This notebook is an AI-assisted portfolio extension of ECE 657 coursework, not an original submitted assignment. Run all cells from top to bottom after installing the repository requirements.

In [ ]:
from pathlib import Path
import sys
root = Path.cwd()
if not (root / 'projects').is_dir(): root = root.parent
sys.path.insert(0, str(root))
import numpy as np
import matplotlib.pyplot as plt

## 1. Penalized logistic likelihood
Let $z=X\\beta$ and $p=\\sigma(z)$. Maximize

$$J=\\frac{1}{N}\\sum_i [y_i z_i-\\log(1+e^{z_i})]-\\lambda_1\\|\\beta_{1:}\\|_1-\\lambda_2\\|\\beta_{1:}\\|_2^2.$$

The intercept is excluded from penalties. The smooth likelihood gradient is $X^T(y-p)/N$; add the L1 subgradient and L2 gradient for penalized coefficients.

In [ ]:
from projects.fundamentals import objective, gradient, split_gain
import inspect
print(inspect.getsource(gradient))
X=np.column_stack([np.ones(5),np.arange(1,6)])
y=np.array([0,0,1,1,1]); b=np.array([-.7697,.5005]); h=1e-5
analytic=gradient(b,X,y)
numeric=np.array([(objective(b+h*v,X,y)-objective(b-h*v,X,y))/(2*h) for v in np.eye(2)])
print('Objective:',objective(b,X,y))
print('Analytical:',analytic,'Finite differences:',numeric)
np.testing.assert_allclose(analytic,numeric,atol=1e-8)

## 2. Decision tree split selection
Gini impurity is $G=1-\\sum_c p_c^2$. Select the threshold that maximizes parent impurity minus weighted child impurity.

In [ ]:
feature=np.arange(6); labels=np.array([0,0,0,1,1,1]); thresholds=np.arange(.5,5,.999999999)
gains=[split_gain(feature,labels,t) for t in thresholds]
plt.bar(thresholds,gains,width=.65); plt.xlabel('Threshold'); plt.ylabel('Gini gain'); plt.show()
print('Best threshold:',thresholds[np.argmax(gains)])

## 3. Interpretation
The analytical gradient agrees with a finite-difference approximation away from the L1 kink. The best tree split is the one that most reduces weighted class impurity, not necessarily the one that creates equally sized groups.

**Practice:** Increase the penalties and examine their effect on coefficients. Explain why ordinary differentiation fails at zero for L1.